In [ ]:
!pip install transformers datasets sklearn torch -q

In [4]:
!pip install evaluate

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import json
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
import evaluate

# ✅ 데이터 로드
with open("intent_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ✅ 라벨 매핑
label_map = {
    "cinema_location": 0,
    "showtime": 1,
    "movie_info": 2,
    "unknown": 3
}
id_to_label_map = {v: k for k, v in label_map.items()}

# ✅ 숫자 라벨 대응 처리
processed_data = []
for d in data:
    label = d["label"]
    if isinstance(label, int):
        label = id_to_label_map.get(label, "unknown")
    processed_data.append({
        "text": d["text"],
        "label": label_map[label]
    })

# ✅ train/test split
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

# ✅ 토크나이저 및 전처리
tokenizer = BertTokenizerFast.from_pretrained("beomi/kcbert-base")

def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=64)

train_ds = train_ds.map(preprocess, batched=True)
val_ds = val_ds.map(preprocess, batched=True)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# ✅ 모델 로드
model = BertForSequenceClassification.from_pretrained("beomi/kcbert-base", num_labels=4)

# ✅ Trainer 설정
training_args = TrainingArguments(
    output_dir="./intent_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# ✅ 정확도 평가 함수
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return metric.compute(predictions=preds, references=labels)

# ✅ Trainer 객체 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# ✅ 학습
trainer.train()

# ✅ 모델 저장
trainer.save_model("./intent_model")
tokenizer.save_pretrained("./intent_model")

print("✅ Intent classification model training complete!")

# ✅ movie_info 세부 분류 함수
def get_movie_info_type(text):
    if any(keyword in text for keyword in ["줄거리", "내용", "스토리"]):
        return "plot"
    elif any(keyword in text for keyword in ["장르", "카테고리", "종류"]):
        return "genre"
    elif any(keyword in text for keyword in ["평점", "점수", "몇 점"]):
        return "rating"
    elif any(keyword in text for keyword in ["감독", "출연", "배우", "감독이 누구"]):
        return "staff"
    else:
        return "unknown"

# ✅ 예측 및 후처리
from torch.nn.functional import softmax

# 예시 문장
example = "어벤져스 줄거리 알려줘"

# 입력 인코딩
inputs = tokenizer(example, return_tensors="pt")

# 모델 예측
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=1)
    label_id = probs.argmax(dim=1).item()

# 라벨 해석
id2label = {v: k for k, v in label_map.items()}
intent = id2label[label_id]
print("🎯 예측된 의도:", intent)

# 세부 분류 (movie_info인 경우)
if intent == "movie_info":
    detail = get_movie_info_type(example)
    print("🔍 movie_info 세부 타입:", detail)

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.646800,0.166694,0.979167
2,0.040900,0.006080,1.000000
3,0.004900,0.003754,1.000000


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Intent classification model training complete!
🎯 예측된 의도: movie_info
🔍 movie_info 세부 타입: plot


In [6]:
# 학습 모델 테스트
from transformers import BertTokenizerFast, BertForSequenceClassification
import torch

# ✅ 모델과 토크나이저 로드
model = BertForSequenceClassification.from_pretrained("./intent_model")
tokenizer = BertTokenizerFast.from_pretrained("./intent_model")

# 라벨 맵 정의 (학습 때와 동일하게)
id_to_label_map = {
    0: "cinema_location",
    1: "showtime",
    2: "movie_info",
    3: "unknown"
}

# movie_info 세부 분류
def get_movie_info_type(text):
    if any(k in text for k in ["줄거리", "내용", "스토리"]):
        return "plot"
    elif any(k in text for k in ["장르", "카테고리", "종류"]):
        return "genre"
    elif any(k in text for k in ["평점", "점수", "몇 점"]):
        return "rating"
    elif any(k in text for k in ["감독", "출연", "배우", "감독이 누구"]):
        return "staff"
    else:
        return "unknown"

# 테스트 예시
def test_intent(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    label_id = logits.argmax().item()
    intent = id_to_label_map[label_id]
    print(f"📌 입력: {text}")
    print(f"🧠 예측된 의도: {intent}")

    if intent == "movie_info":
        detail = get_movie_info_type(text)
        print(f"🔍 movie_info 세부 요청: {detail}")

# ✅ 예시 실행
test_intent("해운대 줄거리 알려줘")
test_intent("강남 롯데시네마 주소 알려줘")
test_intent("광명시 영화관 어디야?")
test_intent("강남 메가박스에서 슈퍼맨 상영시간 보여줘")
test_intent("슈퍼맨 감독 알려줘")
test_intent("슈퍼맨 감독이 누구야?")
test_intent("슈퍼맨 감독은?")

📌 입력: 해운대 줄거리 알려줘
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: plot
📌 입력: 강남 롯데시네마 주소 알려줘
🧠 예측된 의도: cinema_location
📌 입력: 광명시 영화관 어디야?
🧠 예측된 의도: cinema_location
📌 입력: 강남 메가박스에서 슈퍼맨 상영시간 보여줘
🧠 예측된 의도: showtime
📌 입력: 슈퍼맨 감독 알려줘
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: staff
📌 입력: 슈퍼맨 감독이 누구야?
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: staff
📌 입력: 슈퍼맨 감독은?
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: staff


In [14]:
!pip install seqeval

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16183 sha256=70bee3a7f491daae76926facd94da7b9440acaa1eee2a10e57f9e0f0c91b975b
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\5f\b8\73\0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [15]:
# NER 학습
import torch
from transformers import BertTokenizerFast, BertForTokenClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np

# 1) id2label / label2id 정의 (NER 태그)
id2label = {
    0: "O",
    1: "B-REGION",
    2: "I-REGION",
    3: "B-CINEMA",
    4: "I-CINEMA",
    5: "B-MOVIE",
    6: "I-MOVIE",
}
label2id = {v: k for k, v in id2label.items()}

# 2) 준비한 json 데이터 불러오기 (예: 'ner_dataset.json')
# JSON 구조 예시:
# [
#   {"tokens": [...], "ner_tags": [...]},
#   ...
# ]
import json
with open("ner_training_data.json", "r", encoding="utf-8") as f:
    ner_data = json.load(f)

# 3) Dataset 객체 생성
dataset = Dataset.from_list(ner_data)

# 4) 토크나이저 로드
tokenizer = BertTokenizerFast.from_pretrained("beomi/kcbert-base")

# 5) 토큰화 및 라벨 정렬 함수
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        is_split_into_words=True, 
        truncation=True, 
        padding="max_length", 
        max_length=64
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # ignore index for loss calculation
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                # B- 태그가 I- 태그로 변경 (optional)
                if label[word_idx] % 2 == 1:  # odd index means B-
                    label_ids.append(label[word_idx] + 1)
                else:
                    label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 6) 데이터셋에 토큰화 적용
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# 7) train / validation split
train_size = int(0.8 * len(tokenized_dataset))
train_dataset = tokenized_dataset.select(range(train_size))
eval_dataset = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

# 8) 데이터 포맷 지정
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 9) 모델 로드 (num_labels = 태그 개수)
model = BertForTokenClassification.from_pretrained(
    "beomi/kcbert-base", 
    num_labels=len(id2label), 
    id2label=id2label, 
    label2id=label2id
)

# 10) 평가 지표 정의
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)

    # 라벨 -100 제외하고 mapping
    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [id2label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]

    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 11) TrainingArguments 설정
training_args = TrainingArguments(
    output_dir="./ner_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 12) Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# 13) 학습 실행
trainer.train()

# 14) 모델 및 토크나이저 저장
trainer.save_model("./ner_model")
tokenizer.save_pretrained("./ner_model")

print("✅ NER model training complete!")

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.001000,0.000220,1.000000,1.000000,1.000000,1.000000
2,0.004100,0.000098,1.000000,1.000000,1.000000,1.000000
3,0.000200,0.000081,1.000000,1.000000,1.000000,1.000000
4,0.000200,0.000069,1.000000,1.000000,1.000000,1.000000
5,0.000200,0.000066,1.000000,1.000000,1.000000,1.000000


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ NER model training complete!


In [ ]:
# 데이터셋 만들기
import random
import json

regions = ["서울", "부산", "인천", "대구", "광주", "대전", "수원", "강남", "홍대", "강동", "광명"]
cinemas = ["CGV", "메가박스", "롯데시네마", "영화관"]
movies = ["기생충", "아바타", "슈퍼맨", "스파이더맨", "타이타닉", "아이언맨", "어벤져스", "전지적 독자 시점", "킹 오브 킹스"]

templates = [
    ["{region}", "{cinema}", "{movie}", "몇", "시", "해"],
    ["{region}", "{cinema}", "{movie}", "줄거리", "알려줘"],
    ["{region}", "{cinema}", "{movie}", "상영시간", "알려줘"],
    ["{region}", "{cinema}", "{movie}", "감독", "누구야"],
    ["{region}", "{cinema}", "위치", "알려줘"],
    ["{movie}", "평점", "알려줘"],
    ["{region}", "{cinema}", "주소", "알려줘"],
    ["{movie}", "장르", "알려줘"]
    ["{region}", "{cinema}", "위치", "찾아줘"],
    ["{movie}", "평점", "찾아줘"],
    ["{region}", "{cinema}", "주소", "찾아줘"],
    ["{movie}", "장르", "찾아줘"]
    ["{region}", "{cinema}", "위치", "뭐야"],
    ["{movie}", "평점", "뭐야"],
    ["{region}", "{cinema}", "주소", "뭐야"],
    ["{movie}", "장르", "뭐야"]
]

label_map = {
    "O": 0,
    "B-REGION": 1,
    "I-REGION": 2,
    "B-CINEMA": 3,
    "I-CINEMA": 4,
    "B-MOVIE": 5,
    "I-MOVIE": 6
}

def label_tokens(tokens, region, cinema, movie):
    labels = []
    for token in tokens:
        if token == region:
            labels.append(label_map["B-REGION"])
        elif token == cinema:
            labels.append(label_map["B-CINEMA"])
        elif token == movie:
            labels.append(label_map["B-MOVIE"])
        else:
            labels.append(label_map["O"])
    return labels

data = []
for _ in range(300):
    region = random.choice(regions)
    cinema = random.choice(cinemas)
    movie = random.choice(movies)
    template = random.choice(templates)
    tokens = [t.format(region=region, cinema=cinema, movie=movie) for t in template]
    ner_tags = label_tokens(tokens, region, cinema, movie)
    data.append({"tokens": tokens, "ner_tags": ner_tags})

with open("ner_training_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

In [32]:
import json
import random

# JSON 경로는 raw string으로 처리
with open(r"C:\kosmo\data\megabox_cinema_with_showtimes.json", "r", encoding="utf-8") as f:
    cinema_data = json.load(f)

region_set = set()
cinema_set = set()

for item in cinema_data:
    region = item.get("region", [])
    cinema_name = item.get("cinema_name", "")

    if isinstance(region, list):
        region_set.update(region)
    else:
        region_set.add(region)

    if cinema_name:
        cinema_set.add(cinema_name)

regions = list(region_set)
cinemas = list(cinema_set)

movies = ["기생충", "아바타", "슈퍼맨", "스파이더맨", "타이타닉", "아이언맨", "어벤져스", "전지적 독자 시점", "킹 오브 킹스"]

templates = [
    ["{region}", "{cinema}", "{movie}", "몇", "시", "해"],
    ["{region}", "{cinema}", "{movie}", "줄거리", "알려줘"],
    ["{region}", "{cinema}", "{movie}", "상영시간", "알려줘"],
    ["{region}", "{cinema}", "{movie}", "감독", "누구야"],
    ["{region}", "{cinema}", "위치", "알려줘"],
    ["{movie}", "평점", "알려줘"],
    ["{region}", "{cinema}", "주소", "알려줘"],
    ["{movie}", "장르", "알려줘"],
    ["{region}", "{cinema}", "위치", "찾아줘"],
    ["{movie}", "평점", "찾아줘"],
    ["{region}", "{cinema}", "주소", "찾아줘"],
    ["{movie}", "장르", "찾아줘"],
    ["{region}", "{cinema}", "위치", "뭐야"],
    ["{movie}", "평점", "뭐야"],
    ["{region}", "{cinema}", "주소", "뭐야"],
    ["{movie}", "장르", "뭐야"]
]

label_map = {
    "O": 0,
    "B-REGION": 1,
    "I-REGION": 2,
    "B-CINEMA": 3,
    "I-CINEMA": 4,
    "B-MOVIE": 5,
    "I-MOVIE": 6
}

def label_tokens(tokens, region, cinema, movie):
    labels = []
    for token in tokens:
        if token == region:
            labels.append(label_map["B-REGION"])
        elif token == cinema:
            labels.append(label_map["B-CINEMA"])
        elif token == movie:
            labels.append(label_map["B-MOVIE"])
        else:
            labels.append(label_map["O"])
    return labels

data = []
for _ in range(300):
    region = random.choice(regions)
    cinema = random.choice(cinemas)
    movie = random.choice(movies)
    template = random.choice(templates)

    tokens = [t.format(region=region, cinema=cinema, movie=movie) for t in template]
    ner_tags = label_tokens(tokens, region, cinema, movie)
    data.append({"tokens": tokens, "ner_tags": ner_tags})

with open("ner_training_data_from_megabox.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ Megabox 기반 NER 학습 데이터 저장 완료")

✅ Megabox 기반 NER 학습 데이터 저장 완료


In [31]:
import torch
from transformers import BertTokenizerFast, BertForTokenClassification

# 모델과 토크나이저 로드
model_path = "./ner_model"  # 모델 저장된 경로
tokenizer = BertTokenizerFast.from_pretrained(model_path)
model = BertForTokenClassification.from_pretrained(model_path)

# 라벨 매핑
label2id = model.config.label2id
id2label = {v: k for k, v in label2id.items()}

# 테스트 문장
sentence = "강남에 영화관 어디 있어?"

# 토크나이징 (offset_mapping 포함)
inputs = tokenizer(sentence,
                   return_tensors="pt",
                   return_offsets_mapping=True,
                   truncation=True,
                   is_split_into_words=False)

offset_mapping = inputs.pop("offset_mapping")  # forward에 안 쓰니까 제거
input_ids = inputs["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# 모델 추론
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits
predictions = torch.argmax(logits, dim=2)[0].tolist()

# 예측된 레이블 매핑
predicted_labels = [id2label[pred_id] for pred_id in predictions]

# 결과 정리 (offset 기준으로 원문 복원)
ner_results = []
prev_word = ""
for (start, end), token, label in zip(offset_mapping[0].tolist(), tokens, predicted_labels):
    if start == 0 and end == 0:
        continue  # [CLS], [SEP] 등 무시

    word = sentence[start:end]
    if label.startswith("B-"):
        ner_results.append((word, label))
    elif label.startswith("I-") and ner_results:
        prev_entity, prev_label = ner_results[-1]
        if prev_label[2:] == label[2:]:  # 같은 엔티티 범주
            ner_results[-1] = (prev_entity + word, prev_label)
        else:
            ner_results.append((word, label))  # 오류 방지용
    else:
        ner_results.append((word, label))

# 결과 출력
print("\n📌 NER 예측 결과:")
for word, label in ner_results:
    print(f"{word}: {label}")



📌 NER 예측 결과:
강남에: B-REGION
영화: B-CINEMA
관: O
어디: O
있어: O
?: O


In [2]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, BertForTokenClassification
from torch.nn.functional import softmax

# 1. 모델 & 토크나이저 로드
intent_model_path = "./intent_model"
ner_model_path = "./ner_model"

intent_tokenizer = BertTokenizerFast.from_pretrained(intent_model_path)
intent_model = BertForSequenceClassification.from_pretrained(intent_model_path)
intent_model.eval()

ner_tokenizer = BertTokenizerFast.from_pretrained(ner_model_path)
ner_model = BertForTokenClassification.from_pretrained(ner_model_path)
ner_model.eval()

# 2. 의도 라벨 맵
id_to_intent_label = {
    0: "cinema_location",
    1: "showtime",
    2: "movie_info",
    3: "unknown"
}

# 3. NER 라벨 맵
id2label = {
    0: "O",
    1: "B-REGION",
    2: "I-REGION",
    3: "B-CINEMA",
    4: "I-CINEMA",
    5: "B-MOVIE",
    6: "I-MOVIE"
}

# 4. movie_info 세부 분류 함수
def get_movie_info_type(text):
    if any(keyword in text for keyword in ["줄거리", "내용", "스토리"]):
        return "plot"
    elif any(keyword in text for keyword in ["장르", "카테고리", "종류"]):
        return "genre"
    elif any(keyword in text for keyword in ["평점", "점수", "몇 점"]):
        return "rating"
    elif any(keyword in text for keyword in ["감독", "출연", "배우", "감독이 누구"]):
        return "staff"
    else:
        return "unknown"

# 5. subword 토큰과 태그를 실제 개체명 단위로 묶는 함수 (조사/어미 제거 포함)
def merge_subwords(tokens, tags):
    merged_entities = []
    current_entity = ""
    current_tag = None

    # 조사/어미로 추정되는 토큰 리스트 (필요시 더 추가 가능)
    ignore_tokens = ["##에서", "##은", "##는", "##이", "##가", "##을", "##를", "##도", "##와", "##과", "##로", "##으로", "##까지"]

    for token, tag in zip(tokens, tags):
        # 조사/어미 제거
        if token in ignore_tokens:
            continue
        
        if tag.startswith("B-"):
            # 기존 개체 종료, 새 개체 시작
            if current_entity:
                merged_entities.append((current_entity, current_tag))
            current_entity = token.replace("##", "")
            current_tag = tag
        elif tag.startswith("I-") and current_tag and tag[2:] == current_tag[2:]:
            # 같은 종류 개체 이어붙임
            if token.startswith("##"):
                current_entity += token[2:]
            else:
                current_entity += token
        else:
            # O 태그 혹은 다른 개체 시작 => 기존 개체 종료
            if current_entity:
                merged_entities.append((current_entity, current_tag))
                current_entity = ""
                current_tag = None
            # O 태그인 경우 무시 (또는 필요시 처리)

    # 마지막 개체 처리
    if current_entity:
        merged_entities.append((current_entity, current_tag))

    return merged_entities

# 6. 챗봇 처리 함수 (의도 분류 + NER + 후처리)
def chatbot_response(text):
    # 의도 분류
    inputs = intent_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = intent_model(**inputs)
        probs = softmax(outputs.logits, dim=1)
        intent_id = probs.argmax(dim=1).item()
        intent = id_to_intent_label[intent_id]

    # NER 토큰화 및 예측
    inputs_ner = ner_tokenizer(text, return_tensors="pt")
    tokens = ner_tokenizer.tokenize(text)
    with torch.no_grad():
        outputs_ner = ner_model(**inputs_ner)
        predictions = torch.argmax(outputs_ner.logits, dim=2)[0].tolist()

    # [CLS], [SEP] 토큰 제외 (첫, 마지막)
    pred_tags = [id2label.get(p, "O") for p in predictions[1:-1]]

    # 토큰과 태그를 실제 개체명 단위로 묶기 (후처리)
    entities = merge_subwords(tokens, pred_tags)

    # movie_info 세부 분류
    detail = None
    if intent == "movie_info":
        detail = get_movie_info_type(text)

    # 답변 생성
    response = f"의도: {intent}\n"
    if detail:
        response += f"세부 타입: {detail}\n"
    if entities:
        response += "추출된 개체명:\n"
        for ent_text, ent_tag in entities:
            response += f"  {ent_text} -> {ent_tag}\n"

    return response

# 7. 테스트 예시
user_input = "강남 메가박스에서 어벤져스 몇 시 해?"
print(chatbot_response(user_input))

의도: showtime
추출된 개체명:
  강남 -> B-REGION
  메박스 -> B-CINEMA
  어벤져스 -> B-MOVIE



In [ ]:
import json
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, BertForTokenClassification
from torch.nn.functional import softmax

# 1. Megabox JSON 데이터 로드
with open(r"C:\kosmo\data\megabox_cinema_with_showtimes.json", "r", encoding="utf-8") as f:
    cinema_data = json.load(f)

# 2. 모델 로드
intent_model_path = "./intent_model"
intent_tokenizer = BertTokenizerFast.from_pretrained(intent_model_path)
intent_model = BertForSequenceClassification.from_pretrained(intent_model_path)
intent_model.eval()

ner_model_path = "./ner_model"
ner_tokenizer = BertTokenizerFast.from_pretrained(ner_model_path)
ner_model = BertForTokenClassification.from_pretrained(ner_model_path)
ner_model.eval()

# 3. 라벨 맵
intent_id2label = {
    0: "cinema_location",
    1: "showtime",
    2: "movie_info",
    3: "unknown"
}

ner_id2label = {
    0: "O",
    1: "B-REGION",
    2: "I-REGION",
    3: "B-CINEMA",
    4: "I-CINEMA",
    5: "B-MOVIE",
    6: "I-MOVIE"
}

# 4. 개체명 추출
def extract_entities(entities):
    result = {"region": "", "cinema": "", "movie": ""}
    current = {"region": "", "cinema": "", "movie": ""}
    current_label = None

    for token, tag in entities:
        if tag.startswith("B-"):
            current_label = tag[2:].lower()
            current[current_label] = token.replace("##", "")
        elif tag.startswith("I-") and current_label == tag[2:].lower():
            current[current_label] += token.replace("##", "")
        else:
            current_label = None
    result.update(current)
    return result

# 5. 상영시간 찾기
def find_showtimes(region, cinema, movie, data):
    for c in data:
        if region and region not in c.get("region", []):
            continue
        if cinema and cinema not in c.get("cinema_name", ""):
            continue
        if movie:
            for mv in c.get("movies", []):
                if movie in mv.get("title", ""):
                    return mv.get("showtimes", [])
    return []

# 6. 챗봇 응답 함수
def chatbot_response(text):
    print(f"\n[사용자 입력] {text}")

    # 6-1. Intent 분류
    inputs_intent = intent_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs_intent = intent_model(**inputs_intent)
        probs = softmax(outputs_intent.logits, dim=1)
        intent_id = probs.argmax(dim=1).item()
        intent = intent_id2label[intent_id]

    print(f"[의도] {intent}")

    # 6-2. NER 추론
    tokens = ner_tokenizer.tokenize(text)
    inputs_ner = ner_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs_ner = ner_model(**inputs_ner)
        predictions = torch.argmax(outputs_ner.logits, dim=2)[0].tolist()

    # 6-3. 태그 추출
    entities = []
    for token, pred_id in zip(tokens, predictions[1:-1]):
        tag = ner_id2label.get(pred_id, "O")
        if tag != "O":
            entities.append((token, tag))

    print("[개체명]")
    for token, tag in entities:
        print(f"  {token} -> {tag}")

    # 6-4. 엔티티 추출
    extracted = extract_entities(entities)
    region = extracted.get("region", "")
    cinema = extracted.get("cinema", "")
    movie = extracted.get("movie", "")

    # 6-5. 의도별 응답 처리
    if intent == "showtime":
        showtimes = find_showtimes(region, cinema, movie, cinema_data)
        if showtimes:
            times = ", ".join(showtimes)
            return f"{region} {cinema}에서 {movie} 상영시간은 {times} 입니다."
        else:
            return "죄송하지만 해당 상영시간 정보를 찾지 못했습니다."

    elif intent == "cinema_location":
        for c in cinema_data:
            if region and region not in c.get("region", []):
                continue
            if cinema and cinema not in c.get("cinema_name", ""):
                continue
            address = c.get("address", None)
            if address:
                return f"{region} {cinema}의 주소는 {address}입니다."
        return "해당 영화관 정보를 찾지 못했습니다."

    else:
        return "죄송합니다, 해당 질문은 지원하지 않습니다."

# 7. 테스트
if __name__ == "__main__":
    print(chatbot_response("강남 메가박스 어디야?"))
    print(chatbot_response("홍대 메가박스 어디야?"))
    print(chatbot_response("강남 슈퍼맨 몇 시에 해?"))
    print(chatbot_response("홍대 전지적 독자 시점 몇 시에 해?"))


[사용자 입력] 강남 메가박스 어디야?
[의도] cinema_location
[개체명]
  강남 -> B-REGION
  메 -> B-CINEMA
  ##가 -> I-CINEMA
  ##박스 -> I-CINEMA
강남 메가박스의 주소는 서울특별시 서초구 서초대로 77길 3 (서초동) 아라타워 8층입니다.

[사용자 입력] 홍대 메가박스 어디야?
[의도] cinema_location
[개체명]
  홍 -> B-REGION
  ##대 -> I-REGION
  메 -> B-CINEMA
  ##가 -> I-CINEMA
  ##박스 -> I-CINEMA
홍대 메가박스의 주소는 서울특별시 마포구 양화로 147, (동교동) 아일렉스 7층입니다.

[사용자 입력] 강남 슈퍼맨 몇 시에 해?
[의도] showtime
[개체명]
  강남 -> B-REGION
  슈퍼 -> B-MOVIE
  ##맨 -> I-MOVIE
강남 에서 슈퍼맨 상영시간은 14:25 입니다.

[사용자 입력] 홍대 전지적독자시점 몇 시에 해?
[의도] showtime
[개체명]
  홍 -> B-REGION
  ##대 -> I-REGION
  전 -> B-MOVIE
  ##지 -> I-MOVIE
  ##적 -> I-MOVIE
  ##독 -> I-MOVIE
  ##자 -> I-MOVIE
  ##시 -> I-MOVIE
  ##점 -> I-MOVIE
죄송하지만 해당 상영시간 정보를 찾지 못했습니다.
